# Mask R-CNN X101-FPN — Fine-tuning sobre tiras de cómic
**TFG Sergio Salaices — UAH 2026**

Dataset: 320 train / 80 val | 5 clases | Pretrained COCO

In [ ]:
# ── Celda 1: Instalar Detectron2 ──────────────────────────────────────────────
import subprocess, sys, torch

torch_ver  = '.'.join(torch.__version__.split('.')[:2])          # ej. '2.1'
cuda_ver   = torch.version.cuda.replace('.', '')                 # ej. '121'
wheel_url  = (f'https://dl.fbaipublicfiles.com/detectron2/wheels'
              f'/cu{cuda_ver}/torch{torch_ver}/index.html')

print(f'PyTorch {torch_ver} | CUDA {cuda_ver}')
print(f'Probando wheel: {wheel_url}')

res = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'detectron2', '-f', wheel_url, '-q'],
    capture_output=True, text=True
)
if res.returncode == 0:
    print('✓ Detectron2 instalado desde wheel precompilado')
else:
    print('Wheel no disponible — instalando desde fuente (~15 min)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         'git+https://github.com/facebookresearch/detectron2.git', '-q'],
        check=True
    )
    print('✓ Detectron2 instalado desde fuente')

In [ ]:
# ── Celda 2: Imports ──────────────────────────────────────────────────────────
import os, cv2, json, csv, random
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog, MetadataCatalog,
    DatasetMapper, build_detection_train_loader
)
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.utils.visualizer import Visualizer, ColorMode
import detectron2.data.transforms as T

print('Detectron2:', detectron2.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (!)')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), 'GB')

In [ ]:
# ── Celda 3: Localizar dataset ────────────────────────────────────────────────
# Busca instances_train.json y sube UN nivel (annotations/ → dataset/)

BASE = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'instances_train.json' in files:
        candidate = os.path.dirname(root)  # annotations/ → dataset/
        if os.path.isdir(os.path.join(candidate, 'images')):
            BASE = candidate
            break

if BASE is None:
    print('Contenido de /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth < 7:
            print('  ' * depth + os.path.basename(root) + '/')
    raise RuntimeError('Dataset no encontrado — revisa las rutas arriba')

TRAIN_JSON = f'{BASE}/annotations/instances_train.json'
VAL_JSON   = f'{BASE}/annotations/instances_val.json'
TRAIN_IMGS = f'{BASE}/images/train'
VAL_IMGS   = f'{BASE}/images/val'

with open(TRAIN_JSON) as f:
    meta = json.load(f)
cats = {c['id']: c['name'] for c in meta['categories']}

print(f'Dataset en: {BASE}')
print(f'Train: {len(meta["images"])} imgs | Clases: {cats}')

register_coco_instances('comic_train', {}, TRAIN_JSON, TRAIN_IMGS)
register_coco_instances('comic_val',   {}, VAL_JSON,   VAL_IMGS)
print('Datasets registrados ✓')

In [ ]:
# ── Celda 4: Configuración ────────────────────────────────────────────────────
OUTPUT_DIR = '/kaggle/working/mask_rcnn_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    'COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml'))

cfg.DATASETS.TRAIN = ('comic_train',)
cfg.DATASETS.TEST  = ('comic_val',)
cfg.DATALOADER.NUM_WORKERS = 2

# Pesos preentrenados en COCO
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    'COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml')

# Congelar las 2 primeras etapas del backbone (buen truco para datasets pequeños)
cfg.MODEL.BACKBONE.FREEZE_AT = 2

# Solver — 25 épocas sobre 320 imgs con batch=2 → 4000 iters
cfg.SOLVER.IMS_PER_BATCH   = 2
cfg.SOLVER.BASE_LR         = 0.001
cfg.SOLVER.MAX_ITER        = 4000
cfg.SOLVER.STEPS           = (2500, 3500)   # bajadas de LR
cfg.SOLVER.GAMMA           = 0.1
cfg.SOLVER.WARMUP_ITERS    = 300
cfg.SOLVER.WARMUP_FACTOR   = 1.0 / 1000
cfg.SOLVER.CHECKPOINT_PERIOD = 1000
cfg.SOLVER.CLIP_GRADIENTS.ENABLED    = True
cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 5.0

# Cabeza de detección
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES          = 5

# Evaluación cada 1000 iters
cfg.TEST.EVAL_PERIOD = 1000

cfg.OUTPUT_DIR = OUTPUT_DIR

print('Configuración lista')
print(f'  MAX_ITER={cfg.SOLVER.MAX_ITER} | LR={cfg.SOLVER.BASE_LR} | batch={cfg.SOLVER.IMS_PER_BATCH}')
print(f'  FREEZE_AT={cfg.MODEL.BACKBONE.FREEZE_AT} | NUM_CLASSES={cfg.MODEL.ROI_HEADS.NUM_CLASSES}')
print(f'  Salida: {OUTPUT_DIR}')

In [ ]:
# ── Celda 5: Trainer con augmentación ─────────────────────────────────────────
# Augmentaciones pensadas para periódicos históricos (escala variable, contraste desigual)

class ComicTrainer(DefaultTrainer):

    @classmethod
    def build_train_loader(cls, cfg):
        mapper = DatasetMapper(cfg, is_train=True, augmentations=[
            T.RandomFlip(horizontal=True),
            T.ResizeShortestEdge(
                short_edge_length=[480, 512, 544, 576, 608, 640],
                max_size=1000,
                sample_style='choice',
            ),
            T.RandomBrightness(0.8, 1.2),
            T.RandomContrast(0.8, 1.2),
        ])
        return build_detection_train_loader(cfg, mapper=mapper)

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return COCOEvaluator(dataset_name, output_dir=cfg.OUTPUT_DIR)

print('ComicTrainer definido ✓')

In [ ]:
# ── Celda 6: ENTRENAMIENTO ────────────────────────────────────────────────────
# ~1.5-2h en Kaggle GPU
trainer = ComicTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
# ── Celda 7: Evaluación final ─────────────────────────────────────────────────
cfg.MODEL.WEIGHTS = os.path.join(OUTPUT_DIR, 'model_final.pth')
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

predictor  = DefaultPredictor(cfg)
evaluator  = COCOEvaluator('comic_val', output_dir=OUTPUT_DIR)
val_loader = build_detection_test_loader(cfg, 'comic_val')
results    = inference_on_dataset(predictor.model, val_loader, evaluator)

print('\n========== RESULTADOS FINALES ==========')
for task, metrics in results.items():
    print(f'\n[{task}]')
    for k, v in metrics.items():
        print(f'  {k:30s}: {v:.2f}')

In [ ]:
# ── Celda 8: Guardar métricas en CSV ──────────────────────────────────────────
# Formato pensado para luego comparar los 4 modelos del TFG en una misma tabla

CLASSES = ['Titulo_Periodico', 'Fecha', 'Comic', 'Titulo_Comic', 'Autor']
bbox = results.get('bbox', {})
segm = results.get('segm', {})

row = {
    'modelo'        : 'Mask R-CNN X101-FPN',
    'AP_box'        : round(bbox.get('AP',   0), 2),
    'AP50_box'      : round(bbox.get('AP50', 0), 2),
    'AP75_box'      : round(bbox.get('AP75', 0), 2),
    'APs_box'       : round(bbox.get('APs',  0), 2),
    'APm_box'       : round(bbox.get('APm',  0), 2),
    'APl_box'       : round(bbox.get('APl',  0), 2),
    'AP_mask'       : round(segm.get('AP',   0), 2),
    'AP50_mask'     : round(segm.get('AP50', 0), 2),
    'AP75_mask'     : round(segm.get('AP75', 0), 2),
}
for cls in CLASSES:
    row[f'AP_{cls}_box']  = round(bbox.get(f'AP-{cls}', 0), 2)
    row[f'AP_{cls}_mask'] = round(segm.get(f'AP-{cls}', 0), 2)

csv_path = os.path.join(OUTPUT_DIR, 'metricas_maskrcnn.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    writer.writeheader()
    writer.writerow(row)

print(f'Métricas guardadas en: {csv_path}')
print(f"  AP box  : {row['AP_box']}")
print(f"  AP50 box: {row['AP50_box']}")
print(f"  AP mask : {row['AP_mask']}")
print(f"  AP50 msk: {row['AP50_mask']}")

In [ ]:
# ── Celda 9: Visualización de predicciones ────────────────────────────────────
metadata  = MetadataCatalog.get('comic_val')
val_dicts = DatasetCatalog.get('comic_val')
samples   = random.sample(val_dicts, min(6, len(val_dicts)))

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
fig.suptitle('Mask R-CNN X101-FPN — Predicciones sobre val', fontsize=14, fontweight='bold')

for ax, d in zip(axes.flatten(), samples):
    img_bgr = cv2.imread(d['file_name'])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    outputs = predictor(img_bgr)
    instances = outputs['instances'].to('cpu')

    v = Visualizer(img_rgb, metadata=metadata, scale=1.0,
                   instance_mode=ColorMode.SEGMENTATION)
    vis = v.draw_instance_predictions(instances)

    ax.imshow(vis.get_image())
    n = len(instances)
    ax.set_title(f'{os.path.basename(d["file_name"])}  [{n} detecciones]', fontsize=8)
    ax.axis('off')

plt.tight_layout()
vis_path = os.path.join(OUTPUT_DIR, 'predicciones_val.png')
plt.savefig(vis_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en: {vis_path}')

In [ ]:
# ── Celda 10: Listado de archivos de salida ───────────────────────────────────
# Todo lo que está en /kaggle/working se puede descargar desde Output
print('Archivos generados:')
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1024**2
        print(f'  {os.path.relpath(full, OUTPUT_DIR):45s}  {size:7.1f} MB')